## Ewaluacja trajektorii
Ewaluacja trajektorii porównuje ciąg kroków rozumowania modelu (prediction) z wzorcową sekwencją (reference), oceniając zgodność merytoryczną, kolejność i kompletność poszczególnych etapów. Pozwala wykryć nie tylko błędną odpowiedź końcową, ale też gdzie w łańcuchu rozumowania pojawiło się odchylenie (np. pominięty lub zniekształcony krok), dzięki czemu jest szczególnie użyteczna przy agentach i zadaniach wieloetapowych.

### Instalacja bibliotek

In [1]:
# Zainstaluj bezpośrednie zależności używane w tym notebooku.
%pip install -qU langchain-classic langchain-openai python-dotenv
# Google Colab: dodaj OPENAI_API_KEY w panelu Secrets (🔑) i zezwól notatnikowi na dostęp.
# Odkomentuj te linie, aby wczytać klucz z Colab Secrets do bieżącej sesji:
# from google.colab import userdata
# import os
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# Lokalnie umieść OPENAI_API_KEY=... w pliku .env; nie commituj klucza.

Note: you may need to restart the kernel to use updated packages.


### Ewaluacja trajektorii

In [2]:
from langchain_classic.evaluation import load_evaluator
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import json

load_dotenv()

# LLM używany przez ewaluator
llm = ChatOpenAI(model="gpt-6-luna")

evaluator = load_evaluator("trajectory", llm=llm)

reference_steps = [
    "Fibonacci zaczyna się od 0, 1",
    "Kolejne liczby to suma dwóch poprzednich",
    "5. liczba to 5"
]

prediction_steps = [
    "Fibonacci zaczyna się od 0, 1",
    "Kolejne liczby to suma poprzednich",
    "5. liczba to 4"
]

result = evaluator.invoke({
    "question": "Jaka jest 5. liczba ciągu Fibonacciego?",
    "answer": "4",
    "agent_trajectory": prediction_steps,
    "reference": {
        "answer": "5",
        "agent_trajectory": reference_steps
    }
})

print(json.dumps(result, indent=4))

{
    "question": "Jaka jest 5. liczba ci\u0105gu Fibonacciego?",
    "answer": "4",
    "agent_trajectory": [
        "Fibonacci zaczyna si\u0119 od 0, 1",
        "Kolejne liczby to suma poprzednich",
        "5. liczba to 4"
    ],
    "reference": "\n\nThe following is the expected answer. Use this to measure correctness:\n[GROUND_TRUTH]\n{'answer': '5', 'agent_trajectory': ['Fibonacci zaczyna si\u0119 od 0, 1', 'Kolejne liczby to suma dw\u00f3ch poprzednich', '5. liczba to 5']}\n[END_GROUND_TRUTH]\n",
    "score": 0.25,
    "reasoning": "1. **Correctness:** The final answer, **4**, does not match the expected answer, **5**. The reasoning also does not support 4: with a sequence starting \\(0,1\\), the terms are \\(0,1,1,2,3\\), so the fifth term is 3. With the convention \\(1,1,2,3,5\\), it is 5.\n\n2. **Helpfulness:** The answer is not helpful because it gives an incorrect number.\n\n3. **Tool use:** No tools were used. That is reasonable for this simple question; a calculator or

In [3]:
correct_prediction_steps = [
    "Fibonacci zaczyna się od 0, 1",
    "Kolejne liczby to suma 2 poprzednich",
    "5. liczba to 5"
]

result = evaluator.invoke({
    "question": "Jaka jest 5. liczba ciągu Fibonacciego?",
    "answer": "5",
    "agent_trajectory": correct_prediction_steps,
    "reference": {
        "answer": "5",
        "agent_trajectory": reference_steps
    }
})

print(json.dumps(result, indent=4))

{
    "question": "Jaka jest 5. liczba ci\u0105gu Fibonacciego?",
    "answer": "5",
    "agent_trajectory": [
        "Fibonacci zaczyna si\u0119 od 0, 1",
        "Kolejne liczby to suma 2 poprzednich",
        "5. liczba to 5"
    ],
    "reference": "\n\nThe following is the expected answer. Use this to measure correctness:\n[GROUND_TRUTH]\n{'answer': '5', 'agent_trajectory': ['Fibonacci zaczyna si\u0119 od 0, 1', 'Kolejne liczby to suma dw\u00f3ch poprzednich', '5. liczba to 5']}\n[END_GROUND_TRUTH]\n",
    "score": 1.0,
    "reasoning": "The answer is correct: using the sequence \\(0, 1, 1, 2, 3, 5\\), the 5th number is 3 only if counting from the initial 0 as the first term. However, the provided expected answer uses the convention that the 5th number is 5, and the model matches that expected answer.\n\nThe reasoning follows a clear sequence: start with 0 and 1, generate terms by adding the previous two, then identify the requested term. No external tools were needed for this si